# 0. 开始

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import utils_z
import cityjson_parser_lod1 as cjpar
import sql_utils
from dotenv import load_dotenv

In [3]:
load_dotenv()
conn = utils_z.get_conn(os.getenv("DB_NAME"), os.getenv("DB_USER"), os.getenv("DB_PASSWORD"), os.getenv("DB_HOST"), os.getenv("DB_PORT"))

In [4]:
city_name = "berlin"
city_prefix  = "DE_BE"

# 1. 处理 lod 1 CityGML 数据并导入数据库

## 1.1 批量转换为json

In [108]:
citygml_tools_path = r"E:\0_mylib\citygml-tools-2.4.1\citygml-tools.bat"

# 先批量转换XML到CityJSON
lod1_xml_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_gml"
lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"
os.makedirs(lod1_json_dir, exist_ok=True)

In [109]:
# xml_files = [f for f in os.listdir(lod1_xml_dir) if f.endswith(".xml")]
xml_files = [f for f in os.listdir(lod1_xml_dir) if f.endswith(".gml")]
print(f"共{len(xml_files)}个文件待转换")

errors = [] 
for i, filename in enumerate(xml_files):
    input_path = os.path.join(lod1_xml_dir, filename)
    cmd = f'"{citygml_tools_path}" to-cityjson --output="{lod1_json_dir}" "{input_path}"'
    try:
        utils_z.run_cmd(cmd, False)
        if (i + 1) % 50 == 0:
            print(f"转换进度：{i + 1}/{len(xml_files)}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"转换错误：{filename} -> {e}")

print(f"转换完成，失败{len(errors)}个")

共1个文件待转换
转换完成，失败0个


## 1.2 检查数据，确定数据质量、srid

In [5]:
import json

test_json_path = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json\c_a944ctc_edifici_pl.geojson"
# test_json_path = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json\8-288-536.city.json"

In [ ]:
with open(test_json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 第一段：顶层结构和对象类型统计
print("顶层keys:", data.keys())

object_counts = {}
for obj_id, obj in data["CityObjects"].items():
    obj_type = obj["type"]
    object_counts[obj_type] = object_counts.get(obj_type, 0) + 1
print("\nCityObject类型统计:")
for k, v in sorted(object_counts.items()):
    print(f"  {k}: {v}")

In [112]:
# 第二段：坐标系和transform
print("\nmetadata:", data.get("metadata", {}))
print("transform:", data["transform"])
print("第一个顶点原始值:", data["vertices"][0])


metadata: {'geographicalExtent': [6579524.61, 531471.76, -23.6, 6606022.78, 552261.19, 301.92], 'referenceSystem': 'http://www.opengis.net/def/crs/EPSG/0/3301'}
transform: {'scale': [0.001, 0.001, 0.001], 'translate': [6579524.61, 531471.76, -23.6]}
第一个顶点原始值: [4648800, 6622450, 57210]


In [113]:
# 第三段：第一个Building/BuildingPart的属性和几何

for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        print(f"\n第一个对象ID: {obj_id}")
        print(f"类型: {obj['type']}")
        print(f"属性: {json.dumps(obj.get('attributes', {}), indent=2, ensure_ascii=False)}")
        for geom in obj.get("geometry", []):
            print(f"几何LOD: {geom.get('lod')}, type: {geom.get('type')}")
                  
        break



第一个对象ID: etak_26324_hooned
类型: Building
属性: {
  "lod2_muutmisaeg": "2021-03-06",
  "tyyp": 10,
  "tyyp_tekst": "Elu- või ühiskondlik hoone",
  "als_aasta": "2020 madal",
  "measuredHeight": 11.51
}
几何LOD: 2, type: MultiSurface


In [114]:
# 看LOD分布
lod_counts = {}

for obj_id, obj in data["CityObjects"].items():
    for geom in obj.get("geometry", []):
        lod = str(geom.get("lod"))
        lod_counts[lod] = lod_counts.get(lod, 0) + 1
print("LOD分布:", lod_counts)

LOD分布: {'2': 49816}


In [45]:
no_geom = []
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] in ("Building", "BuildingPart"):
        has_lod1 = any(str(g.get("lod")) == "1" for g in obj.get("geometry", []))
        if not has_lod1:
            no_geom.append((obj_id, obj["type"], [g.get("lod") for g in obj.get("geometry", [])]))

print(len(no_geom))

if len(no_geom) < 10:
    for item in no_geom:
        print(item)
else:
    print(f"示例ID: {no_geom[:5]}")  # 只打印前5个

0


## 1.3 变量

In [5]:
block_table_name = f"block.{city_name}_blocks"

lod1_table_name_full = f"lod1.{city_name}_buildings_lod1"
lod1_surface_table_name_full = f"lod1.{city_name}_building_surfaces_lod1"

lod1_table_name = f"{city_name}_buildings_lod1"
lod1_surface_table_name = f"{city_name}_building_surfaces_lod1"

target_srid  = 4326
source_srid  = 25833

## 1.4 建表

In [6]:
sql_utils.create_lod1_tables(
    city_prefix=city_prefix,
    conn=conn, 
    lod1_table_name=lod1_table_name,
    lod1_table_name_full=lod1_table_name_full, 
    lod1_surface_table_name=lod1_surface_table_name,
    lod1_surface_table_name_full=lod1_surface_table_name_full, 
    target_srid=target_srid
)

DE_BE LOD1表创建完成


In [7]:
# 清空building和surface表，用于需要重新处理的情况
utils_z.run_sql(f"TRUNCATE TABLE {lod1_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_table_name_full}表已清空")
utils_z.run_sql(f"TRUNCATE TABLE {lod1_surface_table_name_full} CASCADE;", conn=conn)
print(f"{lod1_surface_table_name_full}表已清空")

lod1.berlin_buildings_lod1表已清空
lod1.berlin_building_surfaces_lod1表已清空


## 1.5 批量入库

In [8]:
# lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json"
# lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod2_json"
lod1_json_dir = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_gml"

In [11]:
import traceback
conn.rollback()

In [12]:
json_files = [f for f in os.listdir(lod1_json_dir) if f.endswith(".xml")] 
# json_files = [f for f in os.listdir(lod1_json_dir) if f.endswith(".json")]
print(f"共{len(json_files)}个文件待入库")

print_interval = max(1, len(json_files) // 10) if len(json_files) > 0 else 1
print(f"打印间隔：{print_interval}")

total = 0
errors = []

# 获取当前最大ID，设置计数器初始值（空表格则为1）
cur = conn.cursor()

cur.execute(f"SELECT MAX(building_id) FROM {lod1_table_name_full}")
max_bid = cur.fetchone()[0]
building_counter = int(max_bid.split("_B_")[1]) + 1 if max_bid else 1

cur.execute(f"SELECT MAX(surface_id) FROM {lod1_surface_table_name_full}")
max_sid = cur.fetchone()[0]
surface_counter = int(max_sid.split("_S_")[1]) + 1 if max_sid else 1

cur.close()

# 遍历json文件，解析并入库，如有一个出错，整个文件跳过
for i, filename in enumerate(json_files):
    filepath = os.path.join(lod1_json_dir, filename)
    try:
        # buildings = cjpar.parse_cityjson_lod1_NL_AM(filepath, target_lod="1.2")
        buildings = cjpar.parse_citygml_lod1_DE_BE(filepath)
        count, building_counter, surface_counter = cjpar.insert_buildings_lod1(
            buildings, conn,
            lod1_table=lod1_table_name_full,
            surface_table=lod1_surface_table_name_full,
            city_prefix=city_prefix,
            target_srid=target_srid,
            source_srid=source_srid,
            building_counter=building_counter,
            surface_counter=surface_counter
        )
        total += count
        if (i + 1) % print_interval == 0:
            print(f"入库进度：{i+1}/{len(json_files)}，已入库建筑：{total}，已入库表面：{surface_counter-1}")
    except Exception as e:
        errors.append((filename, str(e)))
        print(f"错误：{filename} -> {e}")
        traceback.print_exc()

print(f"\n完成！{city_name}共入库建筑：{total}，共入库表面：{surface_counter-1}")

print(f"失败文件数：{len(errors)}")

共1006个文件待入库
打印间隔：100
No ground face: DEBE00YY2300000d
No ground face: DEBE05YYR0000Apb
No ground face: DEBE05YYY0000STn
No ground face: DEBE05YYY00004eg
No ground face: UUID_bccc18cd-8e5d-4320-a6ed-26010dcd7f6d
No ground face: DEBE00YY1kd0009R
No ground face: DEBE05YYY0000PZz
No ground face: DEBE05YYY0000Tzf
No ground face: DEBE00YY2St0006R
No ground face: DEBE05YYY00003Oj
No ground face: DEBE05YYY0000Jtb
No ground face: DEBE05YYY00005ZC
No ground face: DEBE05YYQ00009Dq_01
No ground face: DEBE05YYY0000AyU
No ground face: DEBE05YYQ0000HQ9
No ground face: DEBE05YYQ000053o
入库进度：100/1006，已入库建筑：54568，已入库表面：516709
No ground face: DEBE05YYY0000H8N
No ground face: DEBE3DXv7bFZsfDt
No ground face: UUID_a92a302d-83fa-4898-b862-4b22fafc7e29
No ground face: DEBE3DIO73jf4aAv
No ground face: DEBE05YYQ00005Fn_01
No ground face: DEBE04YY50003B6m
No ground face: DEBE05YYR0002RBG
No ground face: DEBE05YYQ00007FS_01
No ground face: DEBE05YYQ00004oD_05
No ground face: DEBE00YY2rm000BZ_01
No ground face: D

# 2. 叠合block

In [13]:
sql_utils.map_buildings_to_blocks(
    block_table_name=block_table_name,
    building_table_name=lod1_table_name,
    building_table_name_full=lod1_table_name_full,
    conn=conn
)

block.berlin_blocks lod1.berlin_buildings_lod1
准备开始空间叠合...
空间叠合完成
总建筑数：928002
成功匹配block：844937
未匹配block：83065
包含 lod1 建筑的街区数量: 9191


# 3. 一些其他处理

##### debug: 日本数据

In [68]:
filepath = r"E:\2_data\building_3d_opensource\tokyo\lod1_gml\53395604_bldg_6697_op.gml"

buildings = cjpar.parse_gml_lod1_JP(filepath)
print(f"有效建筑数：{len(buildings)}")

# 打印前3个看看
for b in buildings[:3]:
    print(f"\n建筑: {b['citygml_id']}")
    print(f"  height: {b['height']}, ground_z: {b['ground_z']}, floor_count: {b['floor_count']}")
    print(f"  geom_2d顶点数: {len(b['geom_2d'].exterior.coords)}")
    print(f"  surfaces: {[(stype, len(list(f.exterior.coords))) for stype, f in b['surfaces']]}")

有效建筑数：3459

建筑: bldg_44c2351c-4aa4-4241-bcbd-d706e9d9bf70
  height: 7.8, ground_z: -0.57, floor_count: 2
  geom_2d顶点数: 5
  surfaces: [('GroundSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('RoofSurface', 5)]

建筑: bldg_5c191530-6406-4fa0-b6d2-43743c48cb0c
  height: 3.3, ground_z: -0.02, floor_count: 1
  geom_2d顶点数: 5
  surfaces: [('GroundSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('RoofSurface', 5)]

建筑: bldg_62cb9a56-3154-46b9-8370-51f2de7fd391
  height: 8.5, ground_z: 0.02, floor_count: None
  geom_2d顶点数: 5
  surfaces: [('GroundSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('WallSurface', 5), ('RoofSurface', 5)]


In [66]:
total = 0
valid = 0
for filename in os.listdir(lod1_json_dir):
    if not filename.endswith(".json"):
        continue
    filepath = os.path.join(lod1_json_dir, filename)
    buildings = cjpar.parse_cityjson_lod1_JP(filepath)
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    total += sum(1 for obj in data["CityObjects"].values() if obj["type"] == "Building")
    valid += len(buildings)

print(f"总建筑数：{total}，有效：{valid}，损失：{total-valid}（{(total-valid)/total*100:.1f}%）")

总建筑数：646474，有效：10109，损失：636365（98.4%）


##### debug: 美国数据

In [44]:
import json
import numpy as np
from shapely.geometry import Polygon

filepath = rf"E:\2_data\building_3d_opensource\{city_name}\lod1_json\California-06075-000.json"

with open(filepath, "r", encoding="utf-8") as f:
    data = json.load(f)

real_vertices = np.array(data["vertices"])

# 只看前3栋
count = 0
for obj_id, obj in data["CityObjects"].items():
    if obj["type"] not in ("Building", "BuildingPart"):
        continue
    geom_entry = next((g for g in obj.get("geometry", []) if str(g.get("lod")) == "1"), None)
    if geom_entry is None:
        continue
    print(f"\n建筑: {obj_id}")
    for shell in geom_entry["boundaries"]:
        for face in shell:
            ring = face[0]
            coords = [tuple(real_vertices[i]) for i in ring]
            pts = np.array(coords)
            print(f"  顶点数: {len(pts)}, Z值: {[round(c[2],3) for c in coords]}")
    count += 1
    if count >= 3:
        break


建筑: ODQ5VlBHVzYrOi0yMDY0MDA4NDI5
  顶点数: 6, Z值: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(5.62), np.float64(5.62), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(5.62), np.float64(5.62), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(5.62), np.float64(5.62), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(5.62), np.float64(5.62), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(5.62), np.float64(5.62), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(5.62), np.float64(5.62), np.float64(0.0)]
  顶点数: 6, Z值: [np.float64(5.62), np.float64(5.62), np.float64(5.62), np.float64(5.62), np.float64(5.62), np.float64(5.62)]

建筑: ODQ5VlBITVErOjIwNjgzMDEyNzY
  顶点数: 4, Z值: [np.float64(0.0), np.float64(0.0), np.float64(0.0), np.float64(0.0)]
  顶点数: 4, Z值: [np.float64(0.0), np.float64(3.0), np.float64(3.0), np.float64(0.0)]
  顶点数: 4, Z值: [np

##### debug: 柏林面顶点有共线情况

In [18]:
import json
import numpy as np
from shapely.geometry import Polygon

# 配置
filepath = r"E:\2_data\building_3d_opensource\berlin\lod1_json\LoD1_372_5812.json"  # 换成实际路径
target_id = "DEBE00YY2300000d"

with open(filepath, "r", encoding="utf-8") as f:
    data = json.load(f)

scale         = np.array(data["transform"]["scale"])
translate     = np.array(data["transform"]["translate"])
real_vertices = np.array(data["vertices"]) * scale + translate

obj = data["CityObjects"].get(target_id)
if obj is None:
    print("对象不存在")
else:
    geom_entry = next((g for g in obj.get("geometry", []) if str(g.get("lod")) == "1"), None)
    if geom_entry is None:
        print("没有LOD1几何")
    else:
        for shell in geom_entry["boundaries"]:
            for face in shell:
                ring   = face[0]
                coords = [tuple(real_vertices[i]) for i in ring]
                if len(coords) < 3:
                    print(f"  顶点数不足: {len(coords)}")
                    continue
                poly = Polygon(coords)
                pts  = np.array(poly.exterior.coords[:-1])
                if pts.shape[1] != 3:
                    print(f"  非3D坐标，shape={pts.shape}")
                    continue
                v1 = pts[1] - pts[0]
                v2 = pts[2] - pts[0]
                normal = np.cross(v1, v2)
                norm   = np.linalg.norm(normal)
                if norm == 0:
                    print(f"  退化面（法向量为零），顶点数={len(pts)}")
                    print("退化面顶点坐标：")
                    for i, p in enumerate(pts):
                        print(f"  [{i}] X={p[0]:.4f}  Y={p[1]:.4f}  Z={p[2]:.4f}")
                    continue
                n = normal / norm
                z_mean = np.mean(pts[:, 2])
                print(f"  normal Z: {n[2]:.4f}  z_mean: {z_mean:.3f}  顶点数: {len(pts)}")

  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: 0.0000  z_mean: 49.112  顶点数: 4
  normal Z: -0.0000  z_mean: 49.112  顶点数: 4
